# CW_07
# Transmission Matrices

In this exercise, you will implement the methodology required to retrieve the transmission matrix of a multimode fiber modeled with BPM.

In [ ]:
# - No modification necessary -

import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Physical parameters
# -------------------------
wavelength = 633e-9        # meters
k0 = 2 * np.pi / wavelength

# -------------------------
# Grid parameters
# -------------------------
N = 64                    # number of pixels (NxN grid)
dx = 1e-6                  # pixel size (meters)

L = N * dx                 # total size of grid

# Spatial coordinates
x = np.linspace(-L/2, L/2, N)
y = np.linspace(-L/2, L/2, N)
X, Y = np.meshgrid(x, y)

# -------------------------
# Frequency coordinates
# -------------------------
dk = 2 * np.pi / L
kx = np.fft.fftfreq(N, d=dx) * 2 * np.pi
ky = np.fft.fftfreq(N, d=dx) * 2 * np.pi
KX, KY = np.meshgrid(kx, ky)

K2 = KX**2 + KY**2

# -------------------------
# Propagation parameters
# -------------------------
dz = 1e-5                  # propagation step (meters)
num_steps = 200            # number of propagation steps

def plot_field(field, title="Field"):
    intensity = np.abs(field)**2
    phase = np.angle(field)

    fig, axs = plt.subplots(1, 2, figsize=(10, 4))

    im0 = axs[0].imshow(intensity, extent=[x.min(), x.max(), y.min(), y.max()])
    axs[0].set_title(title + " - Intensity")
    plt.colorbar(im0, ax=axs[0])

    im1 = axs[1].imshow(phase, extent=[x.min(), x.max(), y.min(), y.max()], cmap='twilight')
    axs[1].set_title(title + " - Phase")
    plt.colorbar(im1, ax=axs[1])

    plt.tight_layout()
    plt.show()
    
def propagate_bpm(field, index_profile, n_ref):
    """
    Propagate a field through the fiber using BPM
    with a reference refractive index.

    Parameters:
        field (2D array): input field
        index_profile (2D array): refractive index distribution
        n_ref (float): reference index (e.g., cladding index)
    """
    
    field_z = field.copy()
    
    # Reference propagation constant
    k_ref = k0 * n_ref
    
    # Precompute diffraction operator using reference index
    H = np.exp(-1j * K2 * dz / (2 * k_ref))
    
    for _ in range(num_steps):
        
        # Half step diffraction
        field_z = np.fft.ifft2(np.fft.fft2(field_z) * H)
        
        # Phase accumulation from index contrast
        delta_n = index_profile - n_ref
        phase = np.exp(1j * k0 * delta_n * dz)
        field_z *= phase
        
        # Second half step diffraction
        field_z = np.fft.ifft2(np.fft.fft2(field_z) * H)
    
    return field_z

# Part 1

In this part, you will implement a step-index multimode fiber. 

The fiber will have a uniform z-profile, so all you will need do is to create the 2D phase object that represents the fiber core and cladding. For this section, assume that the size of the cladding is significantly larger than the simulation domain, so only the core will appear as a visible feature.

Complete the provided create_fiber_index_profile function, then answer the following questions:
1) What is the V-number of the fiber we have created?
2) Approximately how many modes should this fiber support?

In [ ]:
def create_fiber_index_profile(core_radius, n_core, n_cladding):
    """
    Create a step-index multimode fiber profile.

    Parameters:
        core_radius (float): radius of the fiber core (meters)
        n_core (float): refractive index of core
        n_cladding (float): refractive index of cladding

    Returns:
        n (2D array): refractive index profile
    """
    raise NotImplementedError

In [ ]:
# - No modification necessary -

core_radius = 15e-6
n_core = 1.45
n_cladding = 1.44

fiber_index = create_fiber_index_profile(core_radius, n_core, n_cladding)

plt.figure()
plt.imshow(fiber_index, extent=[x.min(), x.max(), y.min(), y.max()])
plt.title("Fiber Refractive Index Profile")
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.colorbar(label="Refractive Index")
plt.show()

## Discussion
TODO

# Part 2
Now that we can model propagation through our fiber, we must choose a basis with which to excite the fiber. This basis represents the levers we have to control the output light, and should be reflective of the corresponding physical system and the type of control you have over the input light.

For this digital example, we will choose to assume that we have an array of gaussian sources that we can control the amplitude and relative phase of. This could, for example, be a physical array of single mode fibers that is being coupled into a multimode fiber.

You should implement the function generate_input_basis to create this array of gaussian inputs. The positions used in the function should be an evenly spaced square grid of size num_modes_per_axis by num_modes_per_axis that covers the center quarter of the simulation domain. 

In [ ]:
def generate_input_basis(num_modes_per_axis, spot_size):
    """
    Generate a grid of Gaussian input modes.

    Parameters:
        num_modes_per_axis (int): number of modes along x and y
        spot_size (float): width of Gaussian spots

    Returns:
        inputs (list of 2D arrays): list of input fields
        positions (list of tuples): (x0, y0) positions of spots
    """
    
    raise NotImplementedError

In [ ]:
# - No modification necessary -

num_modes_per_axis = 64
spot_size = 2e-6

input_basis, positions = generate_input_basis(num_modes_per_axis, spot_size)

# Plot a few example modes
plot_field(input_basis[0], title="Top left corner")
plot_field(input_basis[64*63], title="Top right corner")
plot_field(input_basis[63], title="Bottom left corner")
plot_field(input_basis[64**2-1], title="Bottom right corner")

# Part 3
Now you will need to calculate the transmission matrix of the fiber.

To do this, begin by implementing the function propagate_all_inputs, which will take each beam in the input basis and propagate it through the fiber. The corresponding outputs should be concatenated together into an array and returned.

After obtaining all the outputs, you can use them to put together the transmission matrix.

The matrix that you want to construct should take an input vector, which encodes a linear combination of elements of the input basis, and output a value for each pixel of the resulting field. This means that it should have shape (num_modes_per_axis)^2 by N^2. Instead of outputting a 2D image, it should output a 1D flattened version of the 2D image that you will reshape. Since we have probed each of our input basis elements separately, the matrix can be trivially constructed.

Implement the function build_transmission_matrix to perform this operation.


In [ ]:
def propagate_all_inputs(input_basis, index_profile):
    """
    Propagate all input modes through the fiber.

    Returns:
        outputs (list of 2D arrays)
    """
    
    raise NotImplementedError

In [ ]:
# - No modification necessary

output_basis = propagate_all_inputs(input_basis, fiber_index)

# Show a few examples
plot_field(input_basis[0], title="Corner input")
plot_field(output_basis[0], title="Corner output")
plot_field(input_basis[64*32+31], title="Center input")
plot_field(output_basis[64*32+31], title="Center output")

In [ ]:
def build_transmission_matrix(inputs, outputs):
    """
    Build transmission matrix T such that:
        output_vector = T @ input_vector
    """
    
    raise NotImplementedError

In [ ]:
# - No modification necessary -

TM = build_transmission_matrix(input_basis, output_basis)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Magnitude
im0 = axes[0].imshow(np.abs(TM), aspect='auto')
axes[0].set_title("Transmission Matrix Magnitude")
axes[0].set_xlabel("Input Mode Index")
axes[0].set_ylabel("Output Pixel Index")
fig.colorbar(im0, ax=axes[0])

# Phase
im1 = axes[1].imshow(np.angle(TM), aspect='auto')
axes[1].set_title("Transmission Matrix Phase")
axes[1].set_xlabel("Input Mode Index")
axes[1].set_ylabel("Output Pixel Index")
fig.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

# Part 4
Now that we have our transmission matrix, let's try using it to create a target output. 

As a simple test case, let's assume that we want to find the best way to focus all our light on the central pixel of the output. To do this, you should create the desired output and apply the inverse of the transmission matrix to return to the input basis. Then use the input basis to reconstruct the input field that creates this output.

Implement this behavior in the functions compute_input_from_output and vector_to_field.

After you have done this, answer the following questions:
1) Why is our reconstruction not perfect? We explicitly calculated the inverse of just the one pixel, yet we do not see that at the output of propagation.
2) What could we change about our physical setup/simulation to get a better reconstruction? How might we accomplish this in the physical setup?

In [ ]:
def compute_input_from_output(TM, target_output):
    """
    Compute input vector that best reproduces a desired output
    for a possibly non-square transmission matrix.
    
    Solves: TM @ x ≈ target_output (least squares)
    """
    raise NotImplementedError

In [ ]:
def vector_to_field(input_vector, input_basis):
    """
    Reconstruct input field from basis and coefficients.
    """
    raise NotImplementedError

In [ ]:
# - No modification necessary -

target_x = N // 2
target_y = N // 2
target_pixel = target_y * N + target_x

target_output = np.zeros((N, N), dtype=complex)
target_output[target_y, target_x] = 1
plot_field(target_output, title=f"Target Output")

input_vector = compute_input_from_output(TM, target_output.flatten())

focus_input_field = vector_to_field(input_vector, input_basis)
plot_field(focus_input_field, title="Optimized Input Field")

fiber_output_field = propagate_bpm(focus_input_field, fiber_index, n_cladding)
plot_field(fiber_output_field, title="Fiber Output Field")

ideal_output_field = (np.matmul(TM, input_vector)).reshape((N, N))
plot_field(ideal_output_field, title="Ideal Output Field")

## Discussion
TODO